In [1]:
# Task 3: Symmetric vs Asymmetric INT8 Quantization

## Objective

#The objective of this task is to implement and compare symmetric and asymmetric INT8 quantization using NumPy.

#Symmetric quantization is commonly used for model weights because they are generally centered around zero. Asymmetric quantization is more suitable for activations, which are often non-negative.

#In this task, both methods are applied to the same tensors and their performance is compared using different error metrics.

import numpy as np

# Weight tensor
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

# Activation tensor
activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

def symmetric_quantize(tensor):
    max_abs = np.max(np.abs(tensor))

    scale = max_abs / 127

    zero_point = 0

    quantized = np.round(tensor / scale)

    quantized = np.clip(quantized, -127, 127)

    return quantized.astype(np.int8), scale, zero_point

def asymmetric_quantize(tensor):

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    scale = (x_max - x_min) / 255

    if scale == 0:
        scale = 1.0

    zero_point = round(-128 - (x_min / scale))
    zero_point = int(np.clip(zero_point, -128, 127))

    quantized = np.round(tensor / scale) + zero_point

    quantized = np.clip(quantized, -128, 127)

    return quantized.astype(np.int8), scale, zero_point

def dequantize(quantized_tensor, scale, zero_point):

    return (quantized_tensor.astype(np.float32) - zero_point) * scale

def calculate_metrics(original, reconstructed, quantized):

    error = original - reconstructed

    mae = np.mean(np.abs(error))

    mse = np.mean(error ** 2)

    max_error = np.max(np.abs(error))

    sat_min = np.sum(quantized == -128)

    sat_max = np.sum(quantized == 127)

    sat_total = sat_min + sat_max

    return mae, mse, max_error, sat_min, sat_max, sat_total

def compare_quantization(name, tensor):

    print("=" * 70)
    print(name)

    # Symmetric
    q_sym, scale_sym, zp_sym = symmetric_quantize(tensor)
    dq_sym = dequantize(q_sym, scale_sym, zp_sym)

    # Asymmetric
    q_asym, scale_asym, zp_asym = asymmetric_quantize(tensor)
    dq_asym = dequantize(q_asym, scale_asym, zp_asym)

    sym_metrics = calculate_metrics(tensor, dq_sym, q_sym)
    asym_metrics = calculate_metrics(tensor, dq_asym, q_asym)

    print("\nOriginal Tensor")
    print(tensor)

    print("\nSymmetric Quantized")
    print(q_sym)

    print("\nSymmetric Dequantized")
    print(dq_sym)

    print("\nAsymmetric Quantized")
    print(q_asym)

    print("\nAsymmetric Dequantized")
    print(dq_asym)

    print("\nSymmetric Metrics")
    print(f"Scale       : {scale_sym}")
    print(f"Zero Point  : {zp_sym}")
    print(f"MAE         : {sym_metrics[0]}")
    print(f"MSE         : {sym_metrics[1]}")
    print(f"Max Error   : {sym_metrics[2]}")
    print(f"Saturation  : Min={sym_metrics[3]}, Max={sym_metrics[4]}, Total={sym_metrics[5]}")

    print("\nAsymmetric Metrics")
    print(f"Scale       : {scale_asym}")
    print(f"Zero Point  : {zp_asym}")
    print(f"MAE         : {asym_metrics[0]}")
    print(f"MSE         : {asym_metrics[1]}")
    print(f"Max Error   : {asym_metrics[2]}")
    print(f"Saturation  : Min={asym_metrics[3]}, Max={asym_metrics[4]}, Total={asym_metrics[5]}")

    return sym_metrics, asym_metrics

weights_sym, weights_asym = compare_quantization("WEIGHTS", weights)

activations_sym, activations_asym = compare_quantization("ACTIVATIONS", activations)

outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

without_outlier = outlier_tensor[:-1]

print("\n\nWITH OUTLIER")
compare_quantization("OUTLIER TENSOR", outlier_tensor)

print("\n\nWITHOUT OUTLIER")
compare_quantization("WITHOUT OUTLIER", without_outlier)

# Observations

#- Symmetric quantization uses a zero point of 0 and is suitable for tensors whose values are centered around zero, such as model weights.
#- Asymmetric quantization calculates both the scale and zero point, making it more suitable for tensors with non-negative values, such as activations.
#- The reconstructed tensors are very close to the original tensors, with only small quantization errors.
#- The outlier experiment shows that a single large value increases the scale, reducing the precision available for the remaining values.
#- After removing the outlier, the quantization error decreases because the available quantization levels are distributed over a smaller value range.

# Conclusion

#In this task, symmetric and asymmetric INT8 quantization methods were implemented and compared. Error metrics such as MAE, MSE, and Maximum Error were calculated for both methods. The comparison showed that symmetric quantization is well suited for weight tensors, while asymmetric quantization performs better for activation tensors with values that are not centered around zero. The outlier experiment also demonstrated how large values can negatively affect quantization accuracy.

WEIGHTS

Original Tensor
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Symmetric Metrics
Scale       : 0.018897637724876404
Zero Point  : 0
MAE         : 0.003700774861499667
MSE         : 2.1637999452650547e-05
Max Error   : 0.007874011993408203
Saturation  : Min=0, Max=0, Total=0

Asymmetric Metrics
Scale       : 0.0172549020498991
Zero Point  : 11
MAE         : 0.003803931176662445
MSE         : 2.1238029148662463e-05
Max Error   : 0.007450997829437256
Saturation  : Min=1, Max=1, Total=2
ACTIVATIONS

Original Tensor
[[0. 

((np.float32(0.0011023671),
  np.float32(2.1080245e-06),
  np.float32(0.0023622215),
  np.int64(0),
  np.int64(1),
  np.int64(1)),
 (np.float32(0.0011764795),
  np.float32(1.9377358e-06),
  np.float32(0.002352938),
  np.int64(1),
  np.int64(1),
  np.int64(2)))